In [11]:
from my_dataset import DriveDataset
import transforms as T
import random
import numpy as np
import torch


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



In [12]:
class SegmentationPresetEval:
    def __init__(self, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
        self.transforms = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])

    def __call__(self, img, target):
        return self.transforms(img, target)

def get_transform(train, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
   
    return SegmentationPresetEval(mean=mean, std=std)


In [13]:
from src import topoUnet
from src import PI_image
def create_model(num_classes):
    # model = UNet(in_channels=3, num_classes=num_classes, base_c=32)
    model = topoUnet.UNetWithPI(in_channels=3, pi_channels = 3,num_classes=num_classes, base_c=32)
    pi_model = PI_image.Image_PINet()
    return model,pi_model

In [ ]:
import os
from train_utils import evaluate
from datasetOberon import OberonDataset

def main():
    set_seed(42)
    data_path = r'./'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch_size=1
    # segmentation nun_classes + background
    num_classes = 2
    # using compute_mean_std.py
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    
    val_dataset = DriveDataset(data_path,
                               train=False,
                               transforms=get_transform(train=False, mean=mean, std=std))

    num_workers = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8])

    val_loader = torch.utils.data.DataLoader(val_dataset,
                                             batch_size=1,
                                             num_workers=num_workers,
                                             pin_memory=True,
                                             collate_fn=val_dataset.collate_fn)
    model, pi_model = create_model(num_classes)

    pi_model.to(device)
    model.to(device)
   
    model_weights_path = './my_idea_final/best_model_epoch406_dice0.821.pth'  # 这里替换为你的模型权重文件路径
    checkpoint = torch.load(model_weights_path, map_location=device)
    model.load_state_dict(checkpoint['model'])
    pi_model.load_state_dict(checkpoint['PI_model'])

    model.eval()  
    with torch.no_grad():
        confmat, dice, auc_roc_score, cl_dice, b0_error, b1_error = evaluate(model, pi_model,val_loader, device=device, num_classes=num_classes)
        
  
        print(f"Dice coefficient: {dice:.6f}")
        print(f"AUC ROC Score is: {auc_roc_score.compute():.6f}")
        print(f"ClDice Score is: {cl_dice:.6f}")
        print(f"betti 0 error is: {b0_error:.6f}")
        print(f"betti 1 error is: {b1_error:.6f}")
        print(f"confmat is: {confmat}")

    print("Testing complete.")


In [15]:
if __name__ == '__main__':
    main()

/home/zhuangzhigao/mambaforge/envs/TopoUnet/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zhuangzhigao/mambaforge/envs/TopoUnet/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Test:  [ 0/20]  eta: 0:00:13    time: 0.6661  data: 0.0093  max mem: 1127
Test: Total time: 0:00:13
Dice coefficient: 0.821244
AUC ROC Score is: 0.977444
ClDice Score is: 0.825649
betti 0 error is: 101.250000
betti 1 error is: 24.950000
confmat is: Global Accuracy: 95.406228
Mean IoU: 82.299268
Testing complete.
